# Hunyuan3D-2.1 baseline — image-to-3D, shape only

This runs the **real** Hunyuan3D-2.1 model (not the local rule-based prototype in `ml/cad/`) on a Kaggle GPU, so you can see actual neural 3D generation quality/speed for comparison.

**Scope of this notebook, deliberately kept small for a first run:**
- Text prompt -> a quick reference image (via a small, fast text-to-image model, `sd-turbo`) -> Hunyuan3D-2.1 **shape generation only** (untextured mesh).
- Texture/PBR painting is **not** included here — it needs ~21GB VRAM on its own (~29GB combined with shape), which doesn't reliably fit Kaggle's single-GPU quota (P100/T4 = 16GB). Shape-only needs ~10GB, which fits comfortably.
- This is the smallest step that gets you a real Hunyuan3D mesh to look at. See `docs/cad-finetuning-technical-report.md` Section 4 for the fine-tuning plan this would feed into later.

**Before running:** in the Kaggle notebook's right-hand settings panel, set **Accelerator -> GPU T4 x2 (or P100)** and **Internet -> On** (Internet requires phone-verifying your Kaggle account once, under kaggle.com/settings). Both are required or the cells below will fail.

I could not execute this notebook myself (no GPU in my environment) — the install steps and API calls are transcribed directly from the official [Hunyuan3D-2.1 repo](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1)'s quick-start and `hy3dshape/minimal_demo.py`, but you'll be the first to actually run it end-to-end. If something breaks, the error message plus that repo's README/issues is the place to look.

## 0. Confirm GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU visible -- set Accelerator to GPU T4 x2 or P100 in the notebook settings panel, then Factory Reset the session."
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

## 1. Install Hunyuan3D-2.1 (shape-only — no texture/paint dependencies)

We only install `hy3dshape`'s own requirements, skip the root repo's `requirements.txt` (it pins CUDA-specific torch builds and China-mirror package indexes we don't need here, and pulls in `bpy`/`deepspeed`/texture deps we're not using), and skip `hy3dpaint`'s custom CUDA rasterizer build entirely since shape-only generation doesn't need it.

In [ ]:
import os
# Always start from a directory that's guaranteed to still exist. Deleting
# your *current* working directory (e.g. if a previous run's `%cd` left the
# kernel sitting inside Hunyuan3D-2.1/) breaks getcwd() for every subprocess
# call afterwards, including git itself -- "Unable to read current working
# directory" is exactly that failure mode.
os.chdir("/kaggle/working")

# Remove any stale/partial clone from a previous failed attempt (e.g. a lost
# internet connection mid-clone leaves a broken, incomplete directory that
# silently blocks every future `git clone` into the same path -- `!` shell
# failures don't stop the notebook, so this can go unnoticed for a while).
if os.path.exists("/kaggle/working/Hunyuan3D-2.1"):
    !rm -rf /kaggle/working/Hunyuan3D-2.1

!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git /kaggle/working/Hunyuan3D-2.1
%cd /kaggle/working/Hunyuan3D-2.1
!pip install -q -r hy3dshape/requirements.txt
!pip install -q diffusers accelerate  # for the text-to-image step below (usually already satisfied by the line above)

# Verify the clone actually succeeded and has the files we need, instead of
# finding out three cells later with a confusing ModuleNotFoundError.
assert os.path.exists("hy3dshape/hy3dshape/rembg.py"), \
    "Clone looks incomplete -- rerun this cell (check the git clone output above for the actual error)."
print("clone OK")

## 1b. (Optional) Hugging Face login

The Hunyuan3D-2.1 weights on Hugging Face may or may not require an account/token depending on current license-gating. If the download in step 3 fails with a "gated repo" / 401 error, come back here:

1. Create a free Hugging Face account and an access token at huggingface.co/settings/tokens.
2. In this Kaggle notebook: **Add-ons -> Secrets -> Add a new secret**, name it `HF_TOKEN`, paste the token.
3. Run the cell below.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(hf_token)
    print("logged in to Hugging Face")
except Exception as e:
    print(f"skipping HF login ({e}) -- fine if the model download below isn't gated")

## 2. Text prompt -> reference image

Hunyuan3D-2.1's shape generator takes an **image**, not text (see `docs/cad-finetuning-technical-report.md` Section 1) — so we generate a quick reference image first with a small, fast text-to-image model (`sd-turbo`, single-step, ~2GB), then free its VRAM before loading Hunyuan3D itself.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

PROMPT = "a mechanical bracket with mounting holes, product photo, plain white background, studio lighting"  # <-- edit this

t2i = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16")
t2i = t2i.to("cuda")
reference_image = t2i(prompt=PROMPT, num_inference_steps=2, guidance_scale=0.0).images[0]
reference_image.save("/kaggle/working/reference_image.png")

del t2i
import gc
gc.collect()
torch.cuda.empty_cache()
reference_image

## 3. Reference image -> 3D shape (Hunyuan3D-2.1)

**If your kernel just restarted (e.g. an out-of-memory message):** you don't need to redo step 2 — `reference_image.png` and the cloned repo/installed packages all persist on disk through a restart. Just rerun cell 1 (fast the second time, everything's already downloaded/cached) and skip straight to this cell. Loading this model's checkpoint is itself the single heaviest RAM moment in the whole notebook (it pulls the full multi-billion-parameter checkpoint into CPU RAM before moving it to the GPU) — if it OOMs again on *this specific cell* even with nothing else loaded, try switching **Accelerator** in the settings panel (e.g. T4 x2 <-> P100) and restarting the session, since Kaggle allocates different system RAM alongside different GPU types.

In [ ]:
import sys, os, importlib

# Evict any stale cached import of hy3dshape (from earlier attempts against a
# broken/incomplete clone in this same kernel session), and invalidate
# Python's import-path cache too -- sys.modules isn't the only thing that
# gets stale after repeatedly deleting/recreating directories mid-process.
for mod_name in list(sys.modules):
    if mod_name == "hy3dshape" or mod_name.startswith("hy3dshape."):
        del sys.modules[mod_name]
importlib.invalidate_caches()

# Match upstream's own usage exactly (`cd hy3dshape && python3 minimal_demo.py`)
# instead of sys.path.insert()-ing while cwd is still the repo root -- with
# cwd at repo root, Jupyter's implicit cwd-on-sys.path entry finds the OUTER
# hy3dshape/ folder (which has no __init__.py) before it finds the real
# inner package one level down, resolving to a hollow namespace package.
os.chdir("/kaggle/working/Hunyuan3D-2.1/hy3dshape")
sys.path.insert(0, os.getcwd())

import hy3dshape
assert hy3dshape.__file__ is not None, "still resolving to a namespace package -- see the diagnostic cell"

from PIL import Image
from hy3dshape.rembg import BackgroundRemover
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

image = Image.open("/kaggle/working/reference_image.png").convert("RGB")
image = BackgroundRemover()(image)  # Hunyuan3D expects an isolated subject, not a scene

shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2.1")

# octree_resolution=128 was tried first for speed and produced a degenerate,
# near-solid blob for a mounting-bracket prompt -- 128**3 sampling is too
# coarse to resolve small/thin features like bolt holes and a thin plate
# cross-section. 256 confirmed real structure (tens of thousands of faces,
# not a primitive shape) on the same input. Note the diffusion-sampling step
# (50 steps) is a separate, fixed ~2 minute cost independent of this value --
# don't expect resolution changes alone to fix overall speed.
#   128 -> ~2.1M points (too coarse, produced a degenerate blob)
#   256 -> ~16.8M points (confirmed real structure)   384 (Hunyuan3D default) -> ~57M
SHAPE_OCTREE_RESOLUTION = 256

import time
t0 = time.perf_counter()
mesh = shape_pipeline(image=image, octree_resolution=SHAPE_OCTREE_RESOLUTION, num_chunks=20000)[0]
print(f"shape generation took {time.perf_counter() - t0:.1f}s")

mesh.export("/kaggle/working/output.glb")
mesh.export("/kaggle/working/output.stl")
print(f"faces: {len(mesh.faces)}  watertight: {mesh.is_watertight}")

## 4. Quick preview (same approach as the local prototype's `preview_mesh.py`)

Renders a static image so you can see the result inline without downloading anything yet.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
ax.add_collection3d(Poly3DCollection(mesh.vertices[mesh.faces], facecolor="#c9d6e3", edgecolor="#3a3a3a", linewidths=0.1))
bounds = mesh.bounds
center = bounds.mean(axis=0)
radius = (bounds[1] - bounds[0]).max() / 2
for axis, c in zip("xyz", center):
    getattr(ax, f"set_{axis}lim")(c - radius, c + radius)
ax.set_box_aspect([1, 1, 1])
ax.view_init(elev=25, azim=-55)
ax.set_axis_off()
fig.savefig("/kaggle/working/preview.png", dpi=150)
plt.show()

## 5. Package everything for download

Zips the reference image, mesh (both formats), and preview into one file so you only need to download once.

In [ ]:
!cd /kaggle/working && zip -j hunyuan3d_output.zip reference_image.png output.glb output.stl preview.png
!ls -la /kaggle/working/hunyuan3d_output.zip

## 6. Getting the files back to your local machine

Two ways, depending on whether this is a quick interactive run or one you want to keep:

**A. Quick, during this session (no need to "Save Version"):**
Open the file browser in the left sidebar of the Kaggle notebook editor, navigate to `/kaggle/working/`, and click the `...` menu next to `hunyuan3d_output.zip` -> **Download**. Unzip it locally and you'll have `output.glb`, `output.stl`, `reference_image.png`, `preview.png`.

**B. Durable (keeps the run attached to a saved notebook version):**
Click **Save Version** (top right) -> **Save & Run All**. Once it finishes, open the notebook's page and go to the **Output** tab -- all files under `/kaggle/working/` are listed there with individual download buttons, and that version stays around even after your session ends.

Once you have `output.stl` locally, you can drop it straight into this project's local viewer, e.g.:
```bash
cd ml/cad
python3 scripts/preview_mesh.py --mesh ~/Downloads/output.stl --out compare.png --elev 89 --azim -90
python3 evaluate.py --meshes ~/Downloads --out kaggle_eval.json   # run the same CAD-quality checks on it
```
That's the natural next step: comparing this real Hunyuan3D output against the local rule-based prototype's output on the same kind of prompt, using the evaluator already built in this repo.